In [1]:
import pandas as pd

In [2]:
raw_holdout_file = "../data/raw/GDPa3_80 IgGs_full.xlsx"

In [3]:
file = pd.ExcelFile(raw_holdout_file)
file.sheet_names


['Definitions',
 'Sequences',
 'Assay Data - tidy format',
 'Assay Data - average',
 'Versioning']

In [4]:
sequence_df = pd.read_excel(file, "Sequences")
sequence_df.shape

(80, 7)

In [5]:
assays_df = pd.read_excel(file, "Assay Data - average")
assays_df.shape

(80, 34)

In [6]:
merged_holdout_df = sequence_df.merge(assays_df, on="antibody_id", how="left")
merged_holdout_df.head()

,antibody_id,hc_subtype,lc_subtype,vh_protein_sequence,lc_protein_sequence,hc_dna_sequence,lc_dna_sequence,hac_rt_avg,hac_rt_replicates,hac_rt_stddev,...,tm1_nanodsf_stddev,tm2_nanodsf_avg,tm2_nanodsf_replicates,tm2_nanodsf_stddev,tm3_nanodsf_avg,tm3_nanodsf_replicates,tm3_nanodsf_stddev,tonset_nanodsf_avg,tonset_nanodsf_replicates,tonset_nanodsf_stddev
0,GDPa3-001,IgG1,Lambda,EVQLVESGGGLVQPGGSLRLSCATSGFTFNRYWMSWVRQAPGKGLE...,SYELTQPSSVSVSPGQTATITCSGDVLAKRYARWFQQKPGQAPVLV...,GAAGTGCAGCTCGTGGAGTCCGGCGGCGGGCTCGTGCAGCCCGGCG...,AGCTACGAGCTGACACAGCCTAGCAGCGTGAGCGTGAGCCCCGGGC...,1.703300,1,NaN,...,0.105357,83.130000,3,0.600999,NaN,0,NaN,63.136667,3,0.055076
1,GDPa3-002,IgG1,Kappa,EVQLVQSGAEVKKPGESLRISCKGSGYSFTSYWISWVRQMPGKGLE...,EIVLTQSPGTLSLSPGERATLSCRASQSVSSIYLAWYQQKPGQAPR...,GAGGTGCAGCTGGTGCAGAGCGGCGCCGAGGTGAAGAAGCCCGGCG...,GAGATTGTGCTCACCCAAAGCCCCGGCACCCTGAGCCTGAGCCCTG...,4.039083,1,NaN,...,0.094516,78.916667,3,0.538733,85.143333,3,0.279702,64.220000,3,0.487750
2,GDPa3-003,IgG1,Lambda,EVQLVESGGGLVQPGRSLRLSCTASGFTFGDYAMNWVRQAPGKGLE...,SYELTQPPSVSVSPGQTARITCSGDALPKKYAYWYQQKSGQAPVQV...,GAGGTGCAGCTGGTGGAGAGCGGCGGGGGCCTGGTGCAGCCCGGCA...,AGCTACGAGCTGACACAGCCCCCTAGCGTGAGCGTGAGCCCCGGGC...,NaN,0,NaN,...,0.020817,69.696667,3,0.118462,80.030000,3,0.400375,55.886667,3,0.035119
3,GDPa3-004,IgG1,Lambda,QVQLQQWGAGLLKASETLSLSCAVYGGSFSGYSWSWIRQPPGKGLE...,NFIMTQPHSVSESPEKTVTISCTRSSGSIASNYVQWYQQRPGSAPT...,CAAGTGCAGCTGCAGCAGTGGGGCGCCGGCCTGCTGAAGGCTAGCG...,AACTTCATCATGACACAGCCCCACAGCGTGAGCGAGAGCCCCGAGA...,4.154983,1,NaN,...,0.040415,79.716667,3,0.431316,NaN,0,NaN,55.900000,3,0.145258
4,GDPa3-005,IgG1,Lambda,EVQVVESGGGLVQPGGSLRLSCAVSGLSFSNYWMNWVRQAPGKGLE...,QAVVTQEPSLTVSPGGTVTLTCGSGTGGVTSGHYPYWFQQMPGQVP...,GAGGTGCAAGTGGTCGAGAGCGGCGGGGGGCTGGTCCAACCTGGGG...,CAAGCCGTGGTGACACAAGAGCCTAGCCTCACCGTCAGCCCCGGCG...,4.059950,1,NaN,...,0.095044,80.966667,3,0.185562,NaN,0,NaN,63.386667,3,0.228108


In [7]:
replicate_std_cols = [
    "hac_rt_replicates",
    "hac_rt_stddev",
    "hic_rt_replicates",
    "hic_rt_stddev",
    "polyreactivity_prscore_cho_replicates",
    "polyreactivity_prscore_cho_stddev",
    "polyreactivity_prscore_ova_replicates",
    "polyreactivity_prscore_ova_stddev",
    "sec_%monomer_replicates",
    "sec_%monomer_stddev",
    "smac_rt_replicates",
    "smac_rt_stddev",
    "titer_replicates",
    "titer_stddev",
    "tm1_nanodsf_replicates",
    "tm1_nanodsf_stddev",
    "tm2_nanodsf_replicates",
    "tm2_nanodsf_stddev",
    "tm3_nanodsf_replicates",
    "tm3_nanodsf_stddev",
    "tonset_nanodsf_replicates",
    "tonset_nanodsf_stddev",
]

drop_extra_assays = [
    "hac_rt_avg",
    "sec_%monomer_avg",
    "tm3_nanodsf_avg",
]

drop_dna = [
    "hc_dna_sequence",
    "lc_dna_sequence",
]

In [8]:
filtered_holdout_df = merged_holdout_df.drop(
    columns=(replicate_std_cols + drop_extra_assays + drop_dna), errors="ignore"
)

filtered_holdout_df.columns

Index(['antibody_id', 'hc_subtype', 'lc_subtype', 'vh_protein_sequence',
       'lc_protein_sequence', 'hic_rt_avg', 'polyreactivity_prscore_cho_avg',
       'polyreactivity_prscore_ova_avg', 'smac_rt_avg', 'titer_avg',
       'tm1_nanodsf_avg', 'tm2_nanodsf_avg', 'tonset_nanodsf_avg'],
      dtype='object')

In [9]:
filtered_holdout_df = filtered_holdout_df.rename(columns={
    "lc_protein_sequence": "vl_protein_sequence",
    "smac_rt_avg": "SMAC",
    "hic_rt_avg": "HIC",
    "polyreactivity_prscore_cho_avg": "PR_CHO",
    "polyreactivity_prscore_ova_avg": "PR_Ova",
    "tm1_nanodsf_avg": "Tm1",
    "tm2_nanodsf_avg": "Tm2",
    "tonset_nanodsf_avg": "Tonset",
    "titer_avg": "Titer",
})

filtered_holdout_df.head()

,antibody_id,hc_subtype,lc_subtype,vh_protein_sequence,vl_protein_sequence,HIC,PR_CHO,PR_Ova,SMAC,Titer,Tm1,Tm2,Tonset
0,GDPa3-001,IgG1,Lambda,EVQLVESGGGLVQPGGSLRLSCATSGFTFNRYWMSWVRQAPGKGLE...,SYELTQPSSVSVSPGQTATITCSGDVLAKRYARWFQQKPGQAPVLV...,2.441717,0.045668,0.023629,2.606667,213.640374,68.700000,83.130000,63.136667
1,GDPa3-002,IgG1,Kappa,EVQLVQSGAEVKKPGESLRISCKGSGYSFTSYWISWVRQMPGKGLE...,EIVLTQSPGTLSLSPGERATLSCRASQSVSSIYLAWYQQKPGQAPR...,2.575050,0.107134,0.111901,2.683125,707.199549,71.376667,78.916667,64.220000
2,GDPa3-003,IgG1,Lambda,EVQLVESGGGLVQPGRSLRLSCTASGFTFGDYAMNWVRQAPGKGLE...,SYELTQPPSVSVSPGQTARITCSGDALPKKYAYWYQQKSGQAPVQV...,2.583367,0.363681,0.068824,2.673750,497.435695,62.276667,69.696667,55.886667
3,GDPa3-004,IgG1,Lambda,QVQLQQWGAGLLKASETLSLSCAVYGGSFSGYSWSWIRQPPGKGLE...,NFIMTQPHSVSESPEKTVTISCTRSSGSIASNYVQWYQQRPGSAPT...,2.966683,0.000000,0.077785,2.808542,403.933252,64.293333,79.716667,55.900000
4,GDPa3-005,IgG1,Lambda,EVQVVESGGGLVQPGGSLRLSCAVSGLSFSNYWMNWVRQAPGKGLE...,QAVVTQEPSLTVSPGGTVTLTCGSGTGGVTSGHYPYWFQQMPGQVP...,3.000033,0.104098,0.100244,2.721458,129.203068,68.923333,80.966667,63.386667


In [10]:
filtered_holdout_df.columns

Index(['antibody_id', 'hc_subtype', 'lc_subtype', 'vh_protein_sequence',
       'vl_protein_sequence', 'HIC', 'PR_CHO', 'PR_Ova', 'SMAC', 'Titer',
       'Tm1', 'Tm2', 'Tonset'],
      dtype='object')

## HOLDOUT DATA DOES NOT INCLUDE "AC-SINS_pH6.0" and "AC-SINS_pH7.4"

Can be considered that these are not in "real world data" may have to remove from training data if not available in holdout

Antibody name is also not in holdout. Can be removed from training, but will not have an effect on model.

In [11]:
if "antibody_name" not in filtered_holdout_df.columns:
    filtered_holdout_df["antibody_name"] = pd.NA

filtered_holdout_df["AC-SINS_pH6.0"] = pd.NA
filtered_holdout_df["AC-SINS_pH7.4"] = pd.NA

In [12]:
training_order = [
    "antibody_id",
    "antibody_name",
    "SMAC",
    "HIC",
    "PR_CHO",
    "PR_Ova",
    "AC-SINS_pH6.0",
    "AC-SINS_pH7.4",
    "Tonset",
    "Tm1",
    "Tm2",
    "vh_protein_sequence",
    "vl_protein_sequence",
    "hc_subtype",
    "lc_subtype",
    "Titer",
]

In [13]:
cleaned_holdout_df = filtered_holdout_df[training_order]

In [14]:
# For later dropping

# cleaned_holdout_df = cleaned_holdout_df.drop(columns=["antibody_name", "AC-SINS_pH6.0", "AC-SINS_pH7.4"])


In [15]:
cleaned_holdout_df.to_csv("../data/test_data/cleaned_holdout_data.csv")

In [16]:
cleaned_holdout_df

,antibody_id,antibody_name,SMAC,HIC,PR_CHO,PR_Ova,AC-SINS_pH6.0,AC-SINS_pH7.4,Tonset,Tm1,Tm2,vh_protein_sequence,vl_protein_sequence,hc_subtype,lc_subtype,Titer
0,GDPa3-001,<NA>,2.606667,2.441717,0.045668,0.023629,<NA>,<NA>,63.136667,68.700000,83.130000,EVQLVESGGGLVQPGGSLRLSCATSGFTFNRYWMSWVRQAPGKGLE...,SYELTQPSSVSVSPGQTATITCSGDVLAKRYARWFQQKPGQAPVLV...,IgG1,Lambda,213.640374
1,GDPa3-002,<NA>,2.683125,2.575050,0.107134,0.111901,<NA>,<NA>,64.220000,71.376667,78.916667,EVQLVQSGAEVKKPGESLRISCKGSGYSFTSYWISWVRQMPGKGLE...,EIVLTQSPGTLSLSPGERATLSCRASQSVSSIYLAWYQQKPGQAPR...,IgG1,Kappa,707.199549
2,GDPa3-003,<NA>,2.673750,2.583367,0.363681,0.068824,<NA>,<NA>,55.886667,62.276667,69.696667,EVQLVESGGGLVQPGRSLRLSCTASGFTFGDYAMNWVRQAPGKGLE...,SYELTQPPSVSVSPGQTARITCSGDALPKKYAYWYQQKSGQAPVQV...,IgG1,Lambda,497.435695
3,GDPa3-004,<NA>,2.808542,2.966683,0.000000,0.077785,<NA>,<NA>,55.900000,64.293333,79.716667,QVQLQQWGAGLLKASETLSLSCAVYGGSFSGYSWSWIRQPPGKGLE...,NFIMTQPHSVSESPEKTVTISCTRSSGSIASNYVQWYQQRPGSAPT...,IgG1,Lambda,403.933252
4,GDPa3-005,<NA>,2.721458,3.000033,0.104098,0.100244,<NA>,<NA>,63.386667,68.923333,80.966667,EVQVVESGGGLVQPGGSLRLSCAVSGLSFSNYWMNWVRQAPGKGLE...,QAVVTQEPSLTVSPGGTVTLTCGSGTGGVTSGHYPYWFQQMPGQVP...,IgG1,Lambda,129.203068
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75,GDPa3-076,<NA>,5.507917,3.175017,0.101984,0.049241,<NA>,<NA>,64.380000,70.903333,82.593333,EVQLVESGGGLVQPGGSLRLSCAASGFTFSSYEMNWVRQAPGKGLE...,SSELTQDPAVSVALGQTVRITCQGDSLRSYYASWYQQKPGQAPVLV...,IgG1,Lambda,503.814827
76,GDPa3-077,<NA>,2.916667,4.800000,0.294533,0.202581,<NA>,<NA>,33.170000,56.450000,58.555000,QGQLVQSASEVKKPGTSVTVSCKASGYSFTSHGVSWVRQAPGQGLE...,LTQSPGTLSLSPGESATLSCRASQRMSSTYLAWYQQRPGQAPRLLM...,IgG1,Kappa,337.399997
77,GDPa3-078,<NA>,3.249583,2.775017,0.269222,0.169774,<NA>,<NA>,63.493333,70.136667,83.190000,EVQLVESGGGLVQPGGSLRLSCAASGFTFSRYWMSWVRQAPGKGLE...,DIQLTQSPSTLSASVGDRVTITCRAARSINGWVAWYQQKPGKAPKP...,IgG1,Kappa,509.818667
78,GDPa3-079,<NA>,2.695000,2.441683,0.432427,0.417067,<NA>,<NA>,65.113333,70.506667,NaN,QITLKESGPALVKPTQTLTLTCTLSGFSVSTSGVGVGWIRQPPGKA...,DIQMTQSPSTLSASVGDRVTITCRASQGISEWLAWYQQRPGKAPNL...,IgG1,Kappa,1366.914722
